# GRU for Time Series Forecasting: Melbourne Daily Minimum Temperatures
Ana Sofía Rentería Palomares

PyTorch implementation using GRU. Given the last **30 days**, the model predicts tomorrow's minimum temperature.

Structure adapted from J. Canary, Recurrent Neural Networks (RNNs) and LSTMs, Medium, with the synthetic data replaced by a real dataset.

## 1. Setup

### 1.1 Libraries and random seeds


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(42)
np.random.seed(42)

### 1.2 Plot style (Plotly + color palette)


In [2]:
import seaborn as sns                       # static plots (used for the GIF)
import plotly.graph_objects as go           # interactive plots
import plotly.io as pio                     # plotly global settings (templates, renderer)
from plotly.subplots import make_subplots   # figures with several panels

# Pastel color palette used in every plot
BLUE       = "#4F86C6"
LIGHT_BLUE = "#A9C8EA"
GREEN      = "#5BB89A"
PURPLE     = "#8C6BC8"
PINK       = "#E07AAE"
ORANGE     = "#F6B68A"   # soft orange

def rgba(hex_color, alpha):
    """Convert '#RRGGBB' into an 'rgba(r, g, b, alpha)' string (color with transparency)."""
    h = hex_color.lstrip("#")                                                       # drop the '#'
    return f"rgba({int(h[0:2], 16)}, {int(h[2:4], 16)}, {int(h[4:6], 16)}, {alpha})"  # hex pairs -> 0-255

# Seaborn style (used by the GIF in section 7)
sns.set_theme(style="whitegrid", context="notebook",                # white background with grid
              palette=[BLUE, GREEN, PURPLE, PINK, ORANGE],          # our colors as default cycle
              rc={"axes.titleweight": "bold", "axes.edgecolor": "#cccccc"})  # bold titles, light borders

# Plotly style (all interactive plots)
pio.templates["pastel"] = go.layout.Template(layout=dict(           # custom template named "pastel"
    colorway=[BLUE, GREEN, PURPLE, PINK, ORANGE],                   # default trace colors
    font=dict(family="Helvetica, Arial, sans-serif", size=13, color="#333"),  # global font
    title=dict(font=dict(size=17), x=0.02),                         # title size, left-aligned
    legend=dict(bgcolor="rgba(0,0,0,0)"),                           # transparent legend background
    hoverlabel=dict(font_size=12),                                  # tooltip font size
    margin=dict(l=60, r=30, t=60, b=50),                            # figure margins (px)
))
pio.templates.default = "plotly_white+pastel"                       # white theme + our template
pio.renderers.default = "vscode+notebook_connected"                 # show plots in VS Code / Jupyter

## 2. Data preprocessing
- Values like `?0.2` lost their minus sign in the original file, so `?` is replaced with `-`.
- Two missing days are filled by interpolation, so each RNN step is exactly one day.
- **Chronological split:** 1981-1989 for training, 1990 for testing.
- **Min-max scaling** to [0, 1], fitted on the training data only.
- **Sliding windows:** 30 days 

### 2.1 Load and clean the data


In [3]:
df = pd.read_csv("daily-minimum-temperatures-in-me.csv", skipfooter=2, engine="python")
df.columns = ["Date", "Temp"]
df["Temp"] = df["Temp"].astype(str).str.replace("?", "-").astype(float)
df["Date"] = pd.to_datetime(df["Date"])
df = df.set_index("Date").asfreq("D").interpolate()

### 2.2 Chronological train / test split


In [4]:
train = df.loc[:"1989", "Temp"].values
test  = df.loc["1990", "Temp"].values

### 2.3 Window size and sliding-window function




In [5]:
WINDOW = 30

def make_windows(series, window):
    X, y = [], []
    for i in range(len(series) - window):
        X.append(series[i:i + window])
        y.append(series[i + window])
    return np.array(X), np.array(y)

### 2.4 Min-max normalization to [0, 1]
Neural networks learn faster and more stably when the inputs are small numbers in a similar range. The min and max are taken **from the training data only**, so no information from 1990 leaks into training. The last 30 days of 1989 are added before the test set so the first window can forecast January 1st. `unscale` converts the forecasts back to °C at the end.

**Process:** min / max from train 

In [6]:
t_min, t_max = train.min(), train.max()
scale   = lambda x: (x - t_min) / (t_max - t_min)
unscale = lambda x: x * (t_max - t_min) + t_min

train_s = scale(train)
test_s  = scale(np.concatenate([train[-WINDOW:], test]))   # last WINDOW days of 1989 feed the first 1990 forecast

### 2.5 Build the windows and convert to PyTorch tensors
PyTorch only accepts tensors, not NumPy arrays. A GRU also expects 3 dimensions: (samples, time steps, features). We have a single feature (the temperature), so `unsqueeze(-1)` adds that last dimension of size 1.

**Process:** build windows (train and test) - convert to tensors - add feature dimension - shape (samples, 30, 1)

In [7]:
X_train, y_train = make_windows(train_s, WINDOW)
X_test,  y_test  = make_windows(test_s, WINDOW)

# PyTorch RNN input shape: (samples, timesteps, features)
X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(-1)
X_test  = torch.tensor(X_test,  dtype=torch.float32).unsqueeze(-1)
y_test  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(-1)

print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: torch.Size([3257, 30, 1]) | Test: torch.Size([365, 30, 1])


### 2.6 Plot: the full series


In [8]:
fig = go.Figure()                                                           # empty interactive figure
fig.add_scatter(x=df.loc[:"1989"].index, y=df.loc[:"1989", "Temp"], name="Train (1981–1989)",  # training line
                line=dict(color=BLUE, width=1))
fig.add_scatter(x=df.loc["1990"].index, y=df.loc["1990", "Temp"], name="Test (1990)",          # test line
                line=dict(color=GREEN, width=1))
fig.add_vline(x=pd.Timestamp("1990-01-01"), line=dict(color=ORANGE, width=2.5, dash="dash"))  # train/test split
fig.update_traces(hovertemplate="%{x|%d %b %Y}<br>%{y:.1f} °C<extra></extra>")   # tooltip: date + temp
fig.update_layout(title="Daily minimum temperature, Melbourne 1981–1990", yaxis_title="°C",
                  height=430, legend=dict(orientation="h", y=1.08, x=1, xanchor="right"),  # horizontal legend, top-right
                  xaxis=dict(rangeslider=dict(visible=True, thickness=0.08)))   # zoom slider under the x-axis
fig.show()                                                                   # draw it

**Interpretation.** The series has a strong **annual seasonality**: summer nights (Jan–Feb) average about 15 °C and winter nights (Jul) about 6.7 °C. There is no visible long-term trend, and 1990 (green) looks like the previous years.

**Conclusion.** Training on 1981–1989 and testing on 1990 is a fair split: the test year behaves like the training years. A 30-day window does not see the whole annual cycle, but it does tell the model where in the season we are and how the last days have behaved, which is what matters for tomorrow.

## 3. Define the GRU model
$$z_t = \sigma(W_z x_t + U_z h_{t-1} + b_z) \qquad r_t = \sigma(W_r x_t + U_r h_{t-1} + b_r)$$
$$\tilde{h}_t = \tanh(W_h x_t + U_h (r_t \odot h_{t-1}) + b_h) \qquad h_t = (1 - z_t)\odot h_{t-1} + z_t \odot \tilde{h}_t$$
$$\hat{y} = W_{hy}h_{30} + b_y$$
One recurrent layer with 16 hidden units, same as the RNN. The **update gate** $z_t$ lets the cell copy $h_{t-1}$ almost unchanged, so information (and gradient) can travel across many time steps.
The gates make the GRU have almost 3× the weights of an RNN with the same hidden size; that is part of how the cell works.

### 3.1 Model architecture
The GRU reads the 30 days one by one and summarizes them in its hidden state (its "memory"). The linear layer turns that summary into a single number: tomorrow's temperature.

input window (30 days) - GRU updates its hidden state 30 times - keep the last hidden state (16) - linear layer - forecast ŷ

In [ ]:
class GRUModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=16, output_size=1): #each time step has 1 feature, the value of the series at that moment; the GRU's "memory" is a vector of 16 numbers; the model outputs 1 number, the forecast.
        super().__init__()
        self.rnn = nn.GRU(input_size, hidden_size, batch_first=True)   # <- the only change vs. the RNN notebook
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, h_n = self.rnn(x)        # out: hidden state at every step
        return self.fc(out[:, -1, :]) # keep only the last one

### 3.2 Create the model, loss function and optimizer
the loss tells the model how wrong it is (MSE punishes big errors more), and the optimizer decides how to change the weights to make that error smaller.

 create the model - choose the loss (MSE) - choose the optimizer (Adam, lr = 0.01) - count parameters

In [ ]:
model = GRUModel()                                          # GRU with 16 hidden units + linear output layer
criterion = nn.MSELoss()                                    # loss: mean squared error (on scaled values)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)   # Adam optimizer, learning rate 0.01. Updates the weights and biases of the model based on the gradients computed during backpropagation.

print(model)                                                # show the layers
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))  # count all weights and biases

GRUModel(
  (rnn): GRU(1, 16, batch_first=True)
  (fc): Linear(in_features=16, out_features=1, bias=True)
)
Trainable parameters: 929


## 4. Train the GRU
Full-batch training with Backpropagation Through Time (PyTorch computes it with `loss.backward()`).

### 4.1 Training loop
the model starts with random weights. In every epoch it compares its forecasts with the real values and adjusts the weights a little to reduce the error.



predict - measure error (loss) - compute gradients (backward) - update weights (step) - repeat 300 times

In [ ]:
EPOCHS = 300                 # number of passes over the training set. Each pass helps it lower the training error. 
losses = []                  # training loss of every epoch (for the plot); A measure of the model's error on the training data.

for epoch in range(EPOCHS):
    model.train()            # training mode
    optimizer.zero_grad()    # reset the gradients from the previous epoch
    loss = criterion(model(X_train), y_train)   # forward pass on all windows (full batch) + MSE
    loss.backward()          # BPTT: gradients through the 30 time steps
    optimizer.step()         # update the weights with Adam
    losses.append(loss.item())                  # save the loss as a Python float
    if (epoch + 1) % 50 == 0:                   # print progress every 50 epochs
        print(f"Epoch {epoch + 1}/{EPOCHS}, Loss: {loss.item():.5f}")

Epoch 50/300, Loss: 0.01352
Epoch 100/300, Loss: 0.00938
Epoch 150/300, Loss: 0.00822
Epoch 200/300, Loss: 0.00801
Epoch 250/300, Loss: 0.00796
Epoch 300/300, Loss: 0.00792


### 4.2 Plot: training loss


In [12]:
epochs = np.arange(1, EPOCHS + 1)                                   # x-axis: 1, 2, ..., EPOCHS
fig = go.Figure(go.Scatter(x=epochs, y=losses, mode="lines", name="Training loss",     # loss curve
                           line=dict(color=PURPLE, width=2.5),
                           fill="tozeroy", fillcolor=rgba(PURPLE, 0.15),             # shade area under the curve
                           hovertemplate="Epoch %{x}<br>MSE %{y:.5f}<extra></extra>"))  # tooltip text
fig.update_layout(title="Training loss", xaxis_title="Epoch", yaxis_title="MSE (scaled)", height=400, width=800)
fig.show()                                                          # draw it

## 5. Evaluate the GRU
Predictions are converted back to °C. As a reference we compare with the **naïve forecast** (tomorrow = today).

### 5.1 Forecast the test year (back to °C)



In [13]:
model.eval()                                        # evaluation mode
with torch.no_grad():                               # no gradients needed for prediction
    pred = unscale(model(X_test).numpy().ravel())   # forecast 1990, flatten to 1D and convert back to °C

### 5.2 Naïve baseline (tomorrow = today)
we need a simple reference to know if the model is really useful. If the GRU can't beat "tomorrow = today", it isn't learning anything valuable.

actual = 1990 temperatures - naïve forecast = previous day's value

In [14]:
actual = test                                        # real temperatures of 1990 (°C)
naive  = np.concatenate([[train[-1]], test[:-1]])    # tomorrow = today: shift the series by one day
                                                     # (first day uses the last value of 1989)

### 5.3 Error metrics

errors (actual - forecast) 

In [15]:
def report(name, y, p):
    """Error metrics of forecast p against actual values y."""
    e = y - p                                          # forecast errors (residuals)
    sse  = np.sum(e ** 2)                              # sum of squared errors
    mse  = np.mean(e ** 2)                             # mean squared error
    rmse = np.sqrt(mse)                                # root MSE, in °C
    mae  = np.mean(np.abs(e))                          # mean absolute error, in °C
    r2   = 1 - sse / np.sum((y - y.mean()) ** 2)       # share of the variance explained
    return {"Model": name, "SSE": sse, "MSE": mse, "RMSE": rmse, "MAE": mae, "R²": r2}

results = pd.DataFrame([report("GRU", actual, pred),             # one row per model
                        report("Naïve (t-1)", actual, naive)]).set_index("Model")
print(results.round(3))
results.to_csv("results_GRU_30.csv")   # used by the comparison cell in the GRU notebook

                  SSE    MSE   RMSE    MAE     R²
Model                                            
GRU          1869.311  5.121  2.263  1.746  0.656
Naïve (t-1)  2434.110  6.669  2.582  2.025  0.552


### 5.4 Plot: actual vs. forecast


In [16]:
dates = df.loc["1990"].index                                        # dates of the test year (x-axis)

fig = go.Figure()                                                   # empty figure
fig.add_scatter(x=dates, y=actual, name="Actual", line=dict(color=BLUE, width=1.2), opacity=0.6)  # real values
fig.add_scatter(x=dates, y=pred, name="GRU forecast", line=dict(color=PINK, width=2.2))          # model forecast
fig.add_scatter(x=dates, y=naive, name="Naïve (t-1)", line=dict(color=ORANGE, width=1.5),        # baseline: yesterday's value
                visible="legendonly")                       # click it in the legend to show it
fig.update_traces(hovertemplate="%{y:.1f} °C")                      # tooltip: temp with 1 decimal
fig.update_layout(title="Test year 1990: actual vs. GRU forecast", yaxis_title="°C", height=430,
                  hovermode="x unified",                            # one tooltip for all lines at that date
                  legend=dict(orientation="h", y=1.08, x=1, xanchor="right"))  # horizontal legend, top-right
fig.show()                                                          # draw it

## 6. Looking inside: does the gradient survive 30 steps?
How much does each of the 30 input days influence the forecast? We compute $\left|\partial \hat{y} / \partial x_t\right|$ for every time step, averaged over the test windows.
In the plain RNN the gradient is multiplied by $W_{hh}$ and $\tanh'$ at every step back in time and vanishes. In the GRU, when $z_t \approx 0$ the path $h_t \approx h_{t-1}$ is almost an identity, so **older days can keep a non-zero influence**.

### 6.1 Compute the influence of each input day


In [17]:
X_grad = X_test.clone().requires_grad_(True)         # copy of the test inputs that tracks gradients
model(X_grad).sum().backward()                       # gradient of every forecast w.r.t. its inputs

influence = X_grad.grad.abs().mean(dim=0).squeeze().numpy()   # |gradient| averaged over windows -> one value per time step
influence = influence / influence.max()              # relative to the most influential day

for lag in [1, 2, 5, 10, 15, 20, 25, 30]:            # print a few lags
    print(f"lag {lag:>3}: relative influence = {influence[WINDOW - lag]:.3f}")   # lag 1 = last position of the window
np.save("influence_GRU_30.npy", influence)   # used by the comparison cell in the GRU notebook

lag   1: relative influence = 1.000
lag   2: relative influence = 0.083
lag   5: relative influence = 0.032
lag  10: relative influence = 0.023
lag  15: relative influence = 0.020
lag  20: relative influence = 0.018
lag  25: relative influence = 0.018
lag  30: relative influence = 0.030


### 6.2 Plot: influence per lag


In [18]:
lags = np.arange(WINDOW, 0, -1)                      # 30 = oldest day, 1 = yesterday
bar_colors = sns.blend_palette([GREEN, BLUE, PURPLE, PINK], n_colors=WINDOW).as_hex()  # color gradient, one per bar

fig = go.Figure(go.Bar(x=lags, y=influence, marker_color=bar_colors,              # one bar per lag
                       hovertemplate="Lag %{x} days<br>Relative influence %{y:.3f}<extra></extra>"))
fig.update_layout(title="GRU: influence of each input day on the forecast", height=420, width=900,
                  xaxis=dict(title="Days before the forecast (lag)",
                             autorange="reversed",          # oldest on the left, yesterday on the right
                             dtick=5),                      # a tick every 5 days
                  yaxis_title="Relative |∂ŷ/∂x|")           # gradient size, 1 = most influential day
fig.show()                                                  # draw it

**Interpretation.** Yesterday (lag 1) dominates the forecast. 
**Conclusion.** The GRU forecast is mostly driven by yesterday's temperature, but the **update gate keeps a path open through all 30 steps**, so the start of the window is not forgotten. This is the opposite of a vanishing gradient.

## 7. Animation: the sliding window in action


### 7.1 Prepare the data for the animation


In [19]:
from matplotlib.animation import FuncAnimation, PillowWriter   # build the animation and save it as GIF

history = np.concatenate([train[-WINDOW:], test])          # raw °C, aligned with X_test windows
hist_dates = pd.date_range(dates[0] - pd.Timedelta(days=WINDOW), periods=len(history), freq="D")  # dates for `history`

STEP = 3                                                   # draw every 3rd day → ~120 frames
frames = list(range(0, len(test), STEP)) + [len(test) - 1] # frame indices (+ the last day so it ends complete)

### 7.2 Build, save and show the animation


In [20]:
fig, (ax_top, ax_win) = plt.subplots(2, 1, figsize=(12, 6.5),                  # 2 panels, one above the other
                                     gridspec_kw={"height_ratios": [1.6, 1], "hspace": 0.45})  # top panel taller

# Top: the whole test year, filled in as the window slides
ax_top.plot(dates, actual, color=LIGHT_BLUE, lw=1.2, label="Actual")            # real 1990 temps (static)
line_pred, = ax_top.plot([], [], color=PINK, lw=1.8, label="GRU forecast")      # forecast line, grows each frame
span = [ax_top.axvspan(hist_dates[0], hist_dates[WINDOW - 1], color=GREEN, alpha=0.2, label="Input window")]  # shaded window (in a list so update() can replace it)
dot, = ax_top.plot([], [], "o", color=PINK, ms=8, mec="white", mew=1.5)         # marker on the current forecast
ax_top.set_xlim(hist_dates[0], dates[-1])                                       # fixed x range
ax_top.set_ylim(min(actual.min(), pred.min()) - 1, max(actual.max(), pred.max()) + 1)  # fixed y range with 1° margin
ax_top.set_ylabel("°C"); ax_top.legend(loc="upper center", ncol=3, frameon=False)
title = ax_top.set_title("")                                                    # title text is set in update()

# Bottom: the input sequence x_1 ... x_WINDOW feeding the GRU
win_lags = np.arange(-WINDOW, 0)                                                # x positions: -30 ... -1
ax_win.axvspan(0.5, 1.5, color=ORANGE, alpha=0.35)        # the forecast day
win_line, = ax_win.plot(win_lags, np.zeros(WINDOW), "-o", color=PURPLE, lw=1.4, ms=3.5, label="Input window")  # the 30 inputs
pt_actual, = ax_win.plot([1], [0], "s", color=BLUE, ms=8, label="Actual")       # real value of the next day (square)
pt_pred,   = ax_win.plot([1], [0], "o", color=PINK, ms=8, mec="white", mew=1.5, label="Forecast ŷ")  # model forecast (circle)
ax_win.set_xlim(-WINDOW - 1, 2); ax_win.set_ylim(ax_top.get_ylim())             # same y scale as the top panel
ax_win.set_title(f"Input window ({WINDOW} days) → next day"); ax_win.set_xlabel("Day relative to forecast")
ax_win.set_ylabel("°C"); ax_win.legend(loc="lower left", ncol=3, fontsize=8, frameon=False)
sns.despine(fig)                                                                # remove top/right borders

def update(i):
    """Redraw everything for test day i (called once per frame)."""
    t_end = WINDOW + i                                       # index in `history` of the forecast day
    window = history[i:t_end]                                # the 30 days the GRU sees
    span[0].remove()                                         # move the shaded input window
    span[0] = ax_top.axvspan(hist_dates[i], hist_dates[t_end - 1], color=GREEN, alpha=0.2)
    line_pred.set_data(dates[:i + 1], pred[:i + 1])          # forecast line up to day i
    dot.set_data([dates[i]], [pred[i]])                      # move the marker to day i
    win_line.set_ydata(window)                               # bottom panel: current input window
    pt_actual.set_ydata([actual[i]]); pt_pred.set_ydata([pred[i]])  # bottom panel: actual vs forecast
    title.set_text(f"Sliding-window GRU forecast — {dates[i]:%d %b %Y}   "
                   f"(actual {actual[i]:.1f} °C, forecast {pred[i]:.1f} °C)")
    return line_pred, dot, win_line, pt_actual, pt_pred      # artists that changed

anim = FuncAnimation(fig, update, frames=frames, interval=100, blit=False)       # call update() for each frame, 100 ms apart
anim.save("gru_30_forecast_animation.gif", writer=PillowWriter(fps=10), dpi=90)  # write the GIF (10 frames/s)
plt.close(fig)                                                                   # don't show the static figure too
print("Saved gru_30_forecast_animation.gif")

Saved gru_30_forecast_animation.gif


### 7.3 Interactive animation (Plotly)


In [21]:
y_range = [min(actual.min(), pred.min()) - 1, max(actual.max(), pred.max()) + 1]  # fixed y range for both panels
win_lags = np.arange(-WINDOW, 0)                                                   # x positions of the inputs: -30 ... -1

def window_shapes(i):
    """Shaded rectangles for frame i: input window (top) and forecast day (bottom)."""
    t_end = WINDOW + i                                   # index in `history` of the forecast day
    return [dict(type="rect", xref="x", yref="y", x0=hist_dates[i], x1=hist_dates[t_end - 1],   # green box on top panel
                 y0=y_range[0], y1=y_range[1], fillcolor=rgba(GREEN, 0.2), line_width=0, layer="below"),
            dict(type="rect", xref="x2", yref="y2", x0=0.5, x1=1.5, y0=y_range[0], y1=y_range[1],  # orange box on bottom panel
                 fillcolor=rgba(ORANGE, 0.35), line_width=0, layer="below")]

def frame_data(i):
    """New x/y for the 5 moving traces at frame i."""
    return [go.Scatter(x=dates[:i + 1], y=pred[:i + 1]),                   # 1 forecast so far
            go.Scatter(x=[dates[i]], y=[pred[i]]),                         # 2 current forecast
            go.Scatter(x=win_lags, y=history[i:WINDOW + i]),               # 3 input window
            go.Scatter(x=[1], y=[actual[i]]),                              # 4 actual next day
            go.Scatter(x=[1], y=[pred[i]])]                                # 5 forecast next day

def frame_title(i):
    """Title with the date, actual and forecast of frame i."""
    return (f"<b>Sliding-window GRU forecast</b><br>"
            f"<span style='font-size:13px'>{dates[i]:%d %b %Y} · actual {actual[i]:.1f} °C · "
            f"forecast {pred[i]:.1f} °C</span>")

fig = make_subplots(rows=2, cols=1, row_heights=[0.62, 0.38], vertical_spacing=0.16,       # 2 stacked panels
                    subplot_titles=("", f"Input window ({WINDOW} days) → next day"))
# Trace 0 (static): actual temps of 1990
fig.add_scatter(x=dates, y=actual, name="Actual", line=dict(color=LIGHT_BLUE, width=1.3), row=1, col=1)
# Trace 1: forecast line (filled by the frames)
fig.add_scatter(x=[], y=[], name="GRU forecast", line=dict(color=PINK, width=2), row=1, col=1)
# Trace 2: marker on the current forecast
fig.add_scatter(x=[], y=[], mode="markers", showlegend=False, row=1, col=1,
                marker=dict(color=PINK, size=11, line=dict(color="white", width=2)))
# Trace 3: the 30-day input window (bottom panel)
fig.add_scatter(x=win_lags, y=history[:WINDOW], mode="lines+markers", name="Input window",
                line=dict(color=PURPLE, width=1.6), marker=dict(size=5), row=2, col=1)
# Trace 4: actual value of the next day (square)
fig.add_scatter(x=[1], y=[actual[0]], mode="markers", name="Actual (next day)", row=2, col=1,
                marker=dict(color=BLUE, size=12, symbol="square"))
# Trace 5: forecast of the next day (circle)
fig.add_scatter(x=[1], y=[pred[0]], mode="markers", name="Forecast ŷ", row=2, col=1,
                marker=dict(color=PINK, size=12, line=dict(color="white", width=2)))
for k, t in enumerate(frame_data(0)):                       # start at the first day
    fig.data[k + 1].update(x=t.x, y=t.y)

# One animation frame per selected day: updates traces 1-5, the shaded boxes and the title
fig.frames = [go.Frame(name=str(i), data=frame_data(i), traces=[1, 2, 3, 4, 5],
                       layout=dict(shapes=window_shapes(i), title_text=frame_title(i)))
              for i in frames]

play = dict(frame=dict(duration=100, redraw=True), transition=dict(duration=0), fromcurrent=True)  # 100 ms per frame
fig.update_layout(
    title=dict(text=frame_title(0), y=0.98, yanchor="top"), shapes=window_shapes(0), height=820,   # initial title and boxes
    hovermode="closest", legend=dict(orientation="h", y=1.02, x=0, xanchor="left", yanchor="bottom"),
    xaxis=dict(range=[hist_dates[0], dates[-1]]), yaxis=dict(range=y_range, title="°C"),        # top panel axes
    xaxis2=dict(range=[-WINDOW - 1, 2], title="Day relative to forecast"),                     # bottom panel axes
    yaxis2=dict(range=y_range, title="°C"),
    updatemenus=[dict(type="buttons", direction="left", x=0, y=-0.09, xanchor="left", yanchor="top",  # Play / Pause buttons
                      showactive=False,
                      buttons=[dict(label="▶ Play", method="animate", args=[None, play]),
                               dict(label="❚❚ Pause", method="animate",
                                    args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
    sliders=[dict(x=0, y=-0.15, len=1, xanchor="left", yanchor="top", pad=dict(t=0, b=10),         # slider to jump to any day
                  currentvalue=dict(prefix="Day: ", font=dict(size=13)),
                  steps=[dict(label=f"{dates[i]:%d %b}", method="animate",
                              args=[[str(i)], dict(frame=dict(duration=0, redraw=True), mode="immediate")])
                         for i in frames])],
    margin=dict(t=130, b=190, l=60, r=30),                  # extra bottom space for buttons + slider
)
fig.show()                                                  # draw it

## 8. RNN vs. GRU (same window, same hyperparameters)
Run `RNN_30_Scarlette.ipynb` first: it saves `influence_RNN_30.npy` and `results_RNN_30.csv`. This cell puts both models side by side.

### 8.1 Plot: influence per lag, RNN vs. GRU


In [22]:
inf_rnn = np.load("influence_RNN_30.npy")              # influence per lag saved by the RNN notebook
inf_gru = np.load("influence_GRU_30.npy")              # influence per lag saved above
lags = np.arange(WINDOW, 0, -1)                        # 30 = oldest day, 1 = yesterday

fig = go.Figure()                                      # empty figure
fig.add_scatter(x=lags, y=inf_rnn, name="RNN", mode="lines+markers",       # RNN curve
                line=dict(color=BLUE, width=2.5), marker=dict(size=6))
fig.add_scatter(x=lags, y=inf_gru, name="GRU", mode="lines+markers",       # GRU curve
                line=dict(color=PINK, width=2.5), marker=dict(size=6))
fig.update_traces(hovertemplate="%{y:.2e}")                                 # tooltip in scientific notation
fig.update_layout(title=f"Influence of each input day on the forecast ({WINDOW}-day window)",
                  height=430, width=950, hovermode="x unified",            # one tooltip for both curves
                  xaxis=dict(title="Days before the forecast (lag)", autorange="reversed", dtick=5),  # yesterday on the right
                  yaxis=dict(title="Relative |∂ŷ/∂x| (log scale)", type="log",  # log scale to see small values
                             range=[-3, 0.05],                 # 0.001 → 1
                             tickvals=[0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1],  # where ticks go
                             ticktext=["0.001", "0.002", "0.005", "0.01", "0.02", "0.05",         # readable labels
                                       "0.1", "0.2", "0.5", "1"]),
                  legend=dict(x=0.02, y=0.98))                             # legend in the top-left corner
fig.show()                                             # draw it

**Interpretation.** On a log scale the RNN curve (blue) is almost a **straight line going down**: its influence decays exponentially with the lag (0.157 to 0.004 from lag 2 to lag 30). This is the vanishing gradient. The GRU (pink) drops faster in the first days but becomes **flat after ~15 days**. The two curves cross around lag 15.**Conclusion.** With the same setup, the GRU keeps about **7× more influence** than the RNN at the start of the window (0.030 vs. 0.004). The update gate protects long-term memory; the plain RNN slowly forgets.

### 8.2 Tables: influence and error metrics


In [23]:
print(f"{'lag':>4} {'RNN':>10} {'GRU':>10}")               # table header
for lag in [1, 2, 5, 10, 15, 20, 25, 30]:                     # relative influence of both models at a few lags
    print(f"{lag:>4} {inf_rnn[WINDOW - lag]:>10.2e} {inf_gru[WINDOW - lag]:>10.2e}")

comparison = pd.concat([pd.read_csv("results_RNN_30.csv", index_col=0).loc[["RNN"]], results])  # RNN row + GRU/naïve rows
print()
print(comparison.round(3))                                    # error metrics side by side

 lag        RNN        GRU
   1   1.00e+00   1.00e+00
   2   1.57e-01   8.32e-02
   5   7.53e-02   3.16e-02
  10   3.45e-02   2.33e-02
  15   2.05e-02   1.99e-02
  20   1.20e-02   1.83e-02
  25   7.30e-03   1.77e-02
  30   4.36e-03   3.05e-02

                  SSE    MSE   RMSE    MAE     R²
Model                                            
RNN          1864.544  5.108  2.260  1.739  0.656
GRU          1869.311  5.121  2.263  1.746  0.656
Naïve (t-1)  2434.110  6.669  2.582  2.025  0.552
